In [ ]:
# ** Fase3_ Capítulo 1_ Ir além 1 - Dashboard Agrícola **
## O objetivo é criar uma dashboard para visualizar:
# a. Níveis de umidade, P, K e pH;
# b. Status da irrigação;
# c.Sugestões de irrigação baseadas em clima.

In [20]:
# ** Código python **
import streamlit as st
import pandas as pd
import plotly.express as px
import oracledb

# Configuração da página do Streamlit
st.set_page_config(page_title="Dashboard Agrícola", layout="wide")
st.title("🌱 Dashboard de Monitoramento e Irrigação")

## ** Conexão com o banco de dados **
@st.cache_data(ttl=60)  # Atualiza os dados automaticamente a cada 60 segundos

def carregar_dados():

    # DIGITE AQUI AS SUAS CREDENCIAIS DA BASF / ORACLE
    usuario = "RM572690"
    senha = "240773"
    host = "oracle.fiap.com.br"
    porta = 1521
    servico = "ORCL"

# Criando a string de conexão (DSN)
dsn_tns = f"""
    (DESCRIPTION= (ADDRESS=(PROTOCOL=TCP)(HOST={host})(PORT={porta}))
        (CONNECT_DATA=(SERVICE_NAME={servico}))
        )"""

# Conecta ao banco
conn = oracledb.connect(user=usuario, password=senha, dsn=dsn_tns)

# Altere os nomes das colunas e da tabela de acordo com o seu banco de dados
query = """
SELECT data_coleta, umidade, fosforo, potassio, ph,
status_irrigacao, temperatura, chuva_prevista
FROM dados_solo
ORDER BY data_coleta DESC
"""

df = pd.read_sql(query, con=conn)
conn.close()
return df

#** Tratamento de erro para o dashboard funcionar mesmo se o banco estiver offline
try:
        df = carregar_dados()
        df['data_coleta'] = pd.to_datetime(df['data_coleta'])
except Exception as e:
        st.sidebar.warning("⚠️ Usando dados simulados (Banco Oracle não configurado ou inacessível).")

        # Dados fictícios para você visualizar o layout imediatamente
        datas = pd.date_range(end=pd.Timestamp.now(), periods=15, freq='D')
        df = pd.DataFrame({
            'data_coleta': datas,
            'umidade': [35, 32, 28, 45, 50, 48, 42, 38, 25, 22, 60, 55, 48, 40, 31],
            'fosforo': [42] * 15,
            'potassio': [210] * 15,
'ph': [6.2, 6.3, 6.1, 6.5, 6.4, 6.2, 6.0, 5.8, 6.2, 6.1, 6.3, 6.2, 6.4, 6.1, 6.0],
        'status_irrigacao': ['Desligado', 'Desligado', 'Ligado', 'Ligado', 'Desligado', 'Desligado', 'Desligado', 'Ligado', 'Ligado', 'Ligado', 'Desligado', 'Desligado', 'Desligado', 'Desligado', 'Ligado'],
        'temperatura': [28, 29, 31, 26, 25, 27, 28, 30, 32, 33, 22, 24, 26, 27, 29],
        'chuva_prevista': [0, 0, 2, 15, 25, 5, 0, 0, 0, 0, 40, 10, 0, 0, 0]
    })

# Pega o registro mais recente para os blocos de destaque (KPIs)
dados_atuais = df.iloc[-1]


2026-05-15 17:57:42.383 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:57:42.384 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:57:42.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:57:42.385 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-15 17:57:42.386 No runtime found, using MemoryCacheStorageManager


NameError: name 'host' is not defined

In [ ]:
# ** Racional de Irrigação **
def gerar_sugestao(umidade, temp, chuva):
    if umidade < 30 and chuva < 5:
        return "🚨 Crítico: Ligar irrigação imediatamente. Solo seco e sem previsão de chuva volumosa.", "error"
    elif umidade < 40 and temp > 30 and chuva < 10:
        return "⚠️ Alerta: Ligar irrigação em nível moderado. Risco de alta evapotranspiração.", "warning"
    elif chuva > 20:
        return "🛑 Desligar: Chuva forte prevista ({:.1f}mm). Risco de encharcar o solo.".format(chuva), "success"
    else:
        return "✅ Normal: Umidade do solo adequada para as condições climáticas atuais.", "info"

sugestao_texto, tipo_alerta = gerar_sugestao(
    dados_atuais['umidade'], dados_atuais['temperatura'], dados_atuais['chuva_prevista']
)

# ** LAYOUT DO DASHBOARD (STREAMLIT) **
# Bloco 1: Status do Sistema e Tomada de Decisão
col_status, col_Aviso = st.columns([1, 2])
with col_status:
    st.metric(label="Status Atual do Sistema", value=dados_atuais['status_irrigacao'])
with col_Aviso:
    st.subheader("💡 Sugestão de Manejo Baseada no Clima")
    if tipo_alerta == "error": st.error(sugestao_texto)
    elif tipo_alerta == "warning": st.warning(sugestao_texto)
    elif tipo_alerta == "success": st.success(sugestao_texto)
    else: st.info(sugestao_texto)

st.markdown("---")

# Bloco 2: Indicadores Atuais de Solo e Clima
st.subheader("📊 Condições do Solo (Última Leitura)")
kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Umidade Atual", f"{dados_atuais['umidade']}%")
kpi2.metric("Fósforo (P)", f"{dados_atuais['fosforo']} mg/dm³")
kpi3.metric("Potássio (K)", f"{dados_atuais['potassio']} mg/dm³")
kpi4.metric("pH do Solo", f"{dados_atuais['ph']}")

st.markdown("---")

# Bloco 3: Gráficos de Histórico
st.subheader("📈 Histórico de Monitoramento")
col_graf1, col_graf2 = st.columns(2)

with col_graf1:
    fig_umidade = px.line(df, x='data_coleta', y='umidade', title="Evolução da Umidade do Solo (%)", markers=True)
    fig_umidade.update_traces(line_color='#0066cc')
    st.plotly_chart(fig_umidade, use_container_width=True)

with col_graf2:
    fig_ph = px.line(df, x='data_coleta', y='ph', title="Histórico do pH do Solo", markers=True)
    fig_ph.update_traces(line_color='#2ca02c')
    st.plotly_chart(fig_ph, use_container_width=True)
